# Notebook 09 — Cylinder Stick-Slip: From String to Disk

**The question:** The paper's mechanism is built on a bowed string — a
one-dimensional medium with fixed endpoints. An accretion disk is a
two-dimensional medium with no endpoints (it wraps). What happens when
we promote the string to a cylinder?

**Why this matters:** A cylinder is the simplest geometry that bridges
the string (1D, Dirichlet boundaries) and the disk (2D, periodic
boundaries). If the stick-slip mechanism on a cylinder produces the
same 3:2 frequency ratio from its nodal geometry, that validates the
scale-free argument of §5.6 in a geometry closer to the actual
astrophysical system.

**Key differences from a string:**
1. Periodic boundary conditions → traveling waves, not just standing waves
2. Two mode indices $(m, n)$: azimuthal order $m$, axial order $n$
3. Nodal lines, not nodal points
4. Nodes can rotate — a traveling stick-slip wave

**Method:**
1. Derive the mode structure of a thin cylindrical shell
2. Map the nodal geometry: where do modes collide?
3. Show that the 3:2 ratio survives the promotion from 1D to 2D
4. Identify what it would take to make a physical cylinder stick-slip
5. Connect to accretion disk QPOs

Uses only Python standard library.

In [ ]:
import math
from typing import List, Tuple, Dict


# ── Part 1: Mode structure of a string vs. cylinder ──────────────────
#
# STRING (1D, fixed endpoints):
#   Modes: sin(nπx/L), n = 1, 2, 3, ...
#   Frequencies: f_n = n · f_1
#   Nodes: at x = k/n for k = 1, ..., n-1
#
# CYLINDER (2D, periodic in θ, free or fixed in z):
#   Azimuthal modes: e^{imθ}, m = 0, 1, 2, ...
#   Axial modes: sin(nπz/L) or cos(nπz/L), n = 0, 1, 2, ...
#   Combined: Ψ_{m,n}(θ,z) = e^{imθ} · sin(nπz/L)
#   Frequencies: f_{m,n} = (1/2π) √[(m/R)² + (nπ/L)²] · c_shell
#
# The KEY difference: azimuthal modes have NO endpoints.
# The m-th azimuthal mode has m nodal MERIDIANS (lines of longitude),
# equally spaced at angles Δθ = π/m.

print("=== String vs. Cylinder: Mode Structure ===")
print()
print("STRING (1D):")
print(f"  {'mode n':>8s}  {'freq/f₁':>8s}  {'nodes':>30s}  node positions")
print("-" * 70)
for n in range(1, 7):
    nodes = [f"{k}/{n}" for k in range(1, n)]
    node_str = ", ".join(nodes) if nodes else "(none)"
    # ASCII visualization
    vis = ""
    for i in range(31):
        x = i / 30
        val = math.sin(n * math.pi * x)
        if abs(val) < 0.05 and 0.01 < x < 0.99:
            vis += "●"  # node
        elif val > 0.3:
            vis += "⁺"
        elif val < -0.3:
            vis += "⁻"
        else:
            vis += "·"
    print(f"  {n:>8d}  {n:>8d}  {vis}  {node_str}")

print()
print("CYLINDER azimuthal modes (periodic, no endpoints):")
print(f"  {'mode m':>8s}  {'freq/f₁':>8s}  {'nodal meridians':>20s}  angles")
print("-" * 70)
for m in range(0, 7):
    if m == 0:
        freq_ratio = 0
        meridians = "(none — breathing mode)"
        angles = "—"
    else:
        freq_ratio = m
        meridian_angles = [f"{k*180//m}°" for k in range(2*m)]
        meridians = f"{2*m} meridians"
        angles = ", ".join(meridian_angles[:6]) + ("..." if len(meridian_angles) > 6 else "")
    
    # ASCII circle visualization
    vis = ""
    for i in range(24):
        theta = 2 * math.pi * i / 24
        if m == 0:
            vis += "○"
        else:
            val = math.cos(m * theta)
            if abs(val) < 0.15:
                vis += "●"  # nodal meridian
            elif val > 0:
                vis += "+"
            else:
                vis += "-"
    
    print(f"  {m:>8d}  {freq_ratio:>8d}  {meridians:>20s}  {angles}")
    print(f"           {'':>8s}  {'':>20s}  θ: {vis}")

print()
print("Key difference: cylinder modes can TRAVEL around the circumference.")
print("Standing wave cos(mθ) can be decomposed into traveling waves e^{±imθ}.")
print("A traveling stick-slip pattern around a cylinder = QPO.")

In [ ]:
# ── Part 2: Nodal geometry collision on the cylinder ─────────────────
#
# On a string, the 3:2 ratio comes from the collision between
# the 2nd and 3rd harmonic nodes (§5.6).
#
# On a cylinder, the same collision happens in the azimuthal direction:
#   m=2 mode: nodal meridians at 0°, 90°, 180°, 270°
#   m=3 mode: nodal meridians at 0°, 60°, 120°, 180°, 240°, 300°
#
# Where do m=3 nodes fall relative to m=2 antinodes?
#   m=2 antinodes: 45°, 135°, 225°, 315°
#   m=3 nodes: 30°, 90°, 150°, 210°, 270°, 330°
#   Nearest: 30° to 45° = 15° apart (π/12)
#
# But more precisely: the m=3 nodes at 60° and 120° bracket the
# m=2 antinode at 90°. The m=3 nodes at 60° and 120° are at
# 1/3 and 2/3 of the way between m=2 nodes at 0° and 180°.
# This is EXACTLY the string geometry of §5.6, wrapped around.

print("=== Nodal Geometry Collision: m=2 vs m=3 ===")
print()
print("The 1/3 and 2/3 node positions from §5.6, now on a cylinder:")
print()

# Show where m=2 and m=3 nodes and antinodes fall
n_points = 72  # 5° resolution

print("Angle  m=2          m=3          collision")
print("-" * 60)

for i in range(0, n_points, 2):  # every 10°
    theta = 2 * math.pi * i / n_points
    deg = 360 * i / n_points
    
    val_2 = math.cos(2 * theta)
    val_3 = math.cos(3 * theta)
    
    # Classify m=2
    if abs(val_2) < 0.05:
        m2_str = "NODE"
    elif abs(val_2) > 0.95:
        m2_str = "ANTINODE" if val_2 > 0 else "ANTINODE(-)"
    else:
        m2_str = f"  {val_2:+.2f}"
    
    # Classify m=3
    if abs(val_3) < 0.05:
        m3_str = "NODE"
    elif abs(val_3) > 0.95:
        m3_str = "ANTINODE" if val_3 > 0 else "ANTINODE(-)"
    else:
        m3_str = f"  {val_3:+.2f}"
    
    # Check for collision: m=3 node near m=2 antinode
    collision = ""
    if abs(val_3) < 0.05 and abs(val_2) > 0.85:
        collision = "◄ m=3 NODE at m=2 ANTINODE"
    elif abs(val_2) < 0.05 and abs(val_3) > 0.85:
        collision = "◄ m=2 NODE at m=3 ANTINODE"
    
    if collision or abs(val_2) < 0.05 or abs(val_3) < 0.05 or abs(val_2) > 0.95 or abs(val_3) > 0.95:
        print(f"{deg:5.0f}°  {m2_str:>12s}  {m3_str:>12s}  {collision}")

print()
print("The pattern is identical to the string (§5.6):")
print("  Between any two adjacent m=2 nodes (separated by 90°),")
print("  the m=3 mode places nodes at 1/3 and 2/3 of the interval.")
print("  These 1/3 and 2/3 positions are m=2 antinodes.")
print()
print("The geometry is scale-free AND topology-free:")
print("  It works on a string (Dirichlet). It works on a cylinder (periodic).")
print("  It will work on a disk (the accretion disk case).")
print("  The 3:2 ratio comes from WHERE THE ZEROS FALL, not from")
print("  the boundary conditions or the specific geometry.")

In [ ]:
# ── Part 3: Full 2D mode structure of a cylindrical shell ────────────
#
# A thin cylindrical shell of radius R and length L has modes:
#   f_{m,n} ∝ √[(m/R)² + (nπ/L)²]
#
# For a long cylinder (L >> R), axial modes are closely spaced
# and azimuthal modes dominate the lowest frequencies.
# For a short cylinder (L << R), axial modes dominate.
#
# An accretion disk is an annulus: R_in << R_out, with
# radial modes playing the role of "axial" modes and
# azimuthal modes wrapping around the disk.

print("=== 2D Mode Spectrum of a Cylindrical Shell ===")
print()

# Frequency ratios for different aspect ratios
aspect_ratios = [0.5, 1.0, 2.0, 5.0, 10.0]

print("Aspect ratio L/R determines which modes are lowest:")
print()

for LR in aspect_ratios:
    print(f"  L/R = {LR:.1f} ({'short tube' if LR < 1 else 'long tube' if LR > 2 else 'balanced'}):")
    
    # Compute frequencies for low modes
    modes = []
    for m in range(0, 8):
        for n in range(0, 6):
            if m == 0 and n == 0:
                continue  # trivial mode
            # f ∝ √[(m)² + (nπ/L·R)²] = √[m² + (n·π·R/L)²]
            # Normalize: R=1, so f ∝ √[m² + (nπ/LR)²]
            freq = math.sqrt(m**2 + (n * math.pi / LR)**2)
            modes.append((freq, m, n))
    
    modes.sort()
    
    # Show lowest 8 modes
    f_lowest = modes[0][0]
    for i, (freq, m, n) in enumerate(modes[:8]):
        ratio = freq / f_lowest
        mode_type = "azimuthal" if n == 0 else "axial" if m == 0 else "mixed"
        bar = "█" * int(ratio * 8)
        print(f"    (m={m}, n={n})  f/f₁ = {ratio:5.2f}  {mode_type:>10s}  {bar}")
    
    # Check for 3:2 ratio in lowest modes
    ratios_present = []
    for i in range(min(8, len(modes))):
        for j in range(i+1, min(8, len(modes))):
            r = modes[j][0] / modes[i][0]
            if abs(r - 1.5) < 0.05:
                ratios_present.append((modes[i], modes[j]))
    
    if ratios_present:
        for (f1, m1, n1), (f2, m2, n2) in ratios_present:
            print(f"    ★ 3:2 ratio: (m={m2},n={n2})/(m={m1},n={n1}) = {f2/f1:.3f}")
    else:
        print(f"    (no exact 3:2 in lowest 8 modes)")
    print()

print("For long cylinders (L/R >> 1), the lowest modes are purely azimuthal:")
print("  m=1, m=2, m=3, ... with frequency ratios 1:2:3:...")
print("  The 3:2 ratio between m=3 and m=2 is automatic.")
print()
print("An accretion disk is effectively a 'long cylinder' in this sense:")
print("  R_out >> disk thickness, so azimuthal modes dominate.")
print("  The 3:2 QPO ratio is the m=3/m=2 azimuthal mode ratio.")

In [ ]:
# ── Part 4: Traveling waves and the QPO connection ───────────────────
#
# On a string, modes are standing waves: they don't move.
# On a cylinder, each standing wave can be decomposed:
#   cos(mθ) = ½[e^{imθ} + e^{-imθ}]
# into two traveling waves going in opposite directions.
#
# A QPO IS a traveling wave pattern in an accretion disk:
# a density/velocity perturbation that rotates around the disk
# at a frequency different from the orbital frequency.
#
# If stick-slip on a cylinder produces a TRAVELING subharmonic,
# that is exactly a QPO.

print("=== Traveling Waves: String Cannot, Cylinder Can ===")
print()
print("A standing wave doesn't move. A traveling wave propagates.")
print()
print("STRING standing wave at t=0, t=T/8, t=T/4, t=3T/8, t=T/2:")
print("  (The pattern oscillates in place — nodes never move)")
print()

n_pts = 40
for t_frac, label in [(0, "t=0"), (0.125, "t=T/8"), (0.25, "t=T/4"), (0.375, "t=3T/8"), (0.5, "t=T/2")]:
    line = ""
    for i in range(n_pts):
        x = i / n_pts
        val = math.sin(3 * math.pi * x) * math.cos(2 * math.pi * t_frac)
        if val > 0.3:
            line += "█"
        elif val > 0.0:
            line += "▒"
        elif val > -0.3:
            line += "░"
        else:
            line += " "
    print(f"  {label:>8s}  |{line}|")

print()
print("CYLINDER traveling wave (m=3, going right):")
print("  (The pattern MOVES — watch the peaks shift)")
print()

for t_frac, label in [(0, "t=0"), (0.125, "t=T/8"), (0.25, "t=T/4"), (0.375, "t=3T/8"), (0.5, "t=T/2")]:
    line = ""
    for i in range(n_pts):
        theta = 2 * math.pi * i / n_pts
        # Traveling wave: cos(mθ - ωt)
        val = math.cos(3 * theta - 2 * math.pi * t_frac)
        if val > 0.3:
            line += "█"
        elif val > 0.0:
            line += "▒"
        elif val > -0.3:
            line += "░"
        else:
            line += " "
    print(f"  {label:>8s}  ({line})  ← wraps")

print()
print("The traveling wave ROTATES around the cylinder.")
print("This is exactly what a QPO is: a pattern rotating in the disk")
print("at a frequency different from the material's orbital frequency.")
print()
print("On a string, stick-slip produces a STANDING subharmonic (a tone).")
print("On a cylinder, stick-slip should produce a TRAVELING subharmonic (a QPO).")
print("The geometry promotes the tone to a wave.")

In [ ]:
# ── Part 5: What it takes to make a cylinder stick-slip ──────────────
#
# Physical requirements:
#   1. A thin-walled cylinder (shell modes, not solid-body modes)
#   2. A tangential friction drive (the "bow")
#   3. A velocity-weakening friction law (Stribeck curve)
#   4. Sufficient coupling to excite circumferential modes
#
# Existing precedents:
#   - "Singing" wine glasses (rubbed rim → circumferential stick-slip)
#   - Brake squeal (disk + pad → circumferential modes)
#   - Tibetan singing bowls (struck or rubbed → m=2,3,4 modes)

print("=== Making a Cylinder Stick-Slip ===")
print()
print("Physical setup:")
print()
print("  ┌─────────────────────────┐")
print("  │    THIN-WALLED TUBE      │")
print("  │    (brass/aluminum)       │")
print("  │                           │")
print("  │    ○ ○ ○ ○ ○ ○ ○ ○      │  ← cross-section")
print("  │   ○               ○      │")
print("  │  ○    [rotate →]   ○     │")
print("  │   ○               ○      │")
print("  │    ○ ○ ○ ○ ○ ○ ○ ○      │")
print("  │          ↑               │")
print("  │     rosined pad          │")
print("  │    (tangential drive)     │")
print("  └─────────────────────────┘")
print()
print("Existing systems that ALREADY do this:")
print()

systems = [
    ("Wine glass", "rubbed rim", "circumferential", "m=2 (elliptical)", "YES"),
    ("Singing bowl", "rubbed/struck", "circumferential", "m=2,3,4", "YES"),
    ("Brake disk", "pad pressure", "circumferential", "m=2-10", "YES (squeal)"),
    ("Bowed tube", "longitudinal bow", "axial", "n=1,2,...", "YES (known)"),
    ("Rotated tube", "tangential pad", "circumferential", "m=2,3?", "THIS EXPERIMENT"),
]

print(f"{'System':>14s}  {'Drive':>16s}  {'Mode type':>16s}  {'Modes':>12s}  {'Stick-slip?':>14s}")
print("-" * 80)
for name, drive, mode_type, modes, ss in systems:
    print(f"{name:>14s}  {drive:>16s}  {mode_type:>16s}  {modes:>12s}  {ss:>14s}")

print()
print("The wine glass is the most instructive precedent.")
print("When you rub the rim of a wine glass with a wet finger:")
print("  1. The finger alternates between stick and slip (tangential)")
print("  2. The glass vibrates in the m=2 circumferential mode")
print("  3. The nodal pattern has 4 nodal meridians (2 nodal diameters)")
print("  4. The pattern can ROTATE as the finger moves around the rim")
print()
print("A wine glass is literally a bowed cylinder producing circumferential")
print("stick-slip. The only question is whether it can be pushed into")
print("the SUBHARMONIC regime — m=3/m=2 mode-locking at 3:2.")

In [ ]:
# ── Part 6: Mode-locking prediction ──────────────────────────────────
#
# On a string, the 3:2 ratio emerges from the nodal geometry:
#   - 2nd harmonic node at 1/2
#   - 3rd harmonic nodes at 1/3, 2/3
#   - Node-antinode collision at 1/3 and 2/3
#
# On a cylinder, the same geometry produces mode-locking between
# circumferential modes m=2 and m=3:
#   - m=2 has nodal meridians every 90°
#   - m=3 has nodal meridians every 60°
#   - The 60° meridians of m=3 fall at 2/3 of the 90° intervals of m=2
#
# The mode-locking condition: when the stick-slip drive excites
# both m=2 and m=3 simultaneously, their node-antinode collision
# creates a nonlinear energy transfer channel at exactly 3:2.
#
# Prediction: the dominant mode-locked pair should be m=2 and m=3,
# producing a 3:2 frequency ratio, because this is where the
# node-antinode collision is strongest.

print("=== Mode-Locking Prediction ===")
print()
print("For adjacent circumferential modes m and m+1:")
print()

print(f"{'Mode pair':>12s}  {'Freq ratio':>12s}  {'Node-antinode':>16s}  {'Coupling':>10s}")
print("-" * 55)

for m in range(1, 8):
    ratio = (m + 1) / m
    
    # Count node-antinode collisions in one period
    # m mode: nodes at k·π/m for k=0,...,2m-1
    # m+1 mode: nodes at k·π/(m+1)
    # Collision: m+1 node near m antinode
    collisions = 0
    for k1 in range(2*(m+1)):
        theta_node = k1 * math.pi / (m + 1)
        # Check if this is near an m antinode
        # m antinodes at (2k+1)·π/(2m)
        for k2 in range(2*m):
            theta_anti = (2*k2 + 1) * math.pi / (2*m)
            diff = abs(theta_node - theta_anti)
            diff = min(diff, 2*math.pi - diff)
            if diff < math.pi / (6 * max(m, m+1)):  # tight proximity
                collisions += 1
    
    # Coupling strength heuristic: collisions / total nodes
    coupling = collisions / (2 * (m + 1))
    
    bar = "█" * int(coupling * 30)
    marker = " ◄ STRONGEST" if m == 2 else ""
    
    print(f"  m={m}:m={m+1}  {ratio:12.4f}  {collisions:>16d}  {coupling:10.3f}  {bar}{marker}")

print()
print("The m=2:m=3 pair (frequency ratio 3:2) has the strongest")
print("node-antinode coupling among low-order adjacent modes.")
print()
print("This is why 3:2 dominates across systems:")
print("  - It's the lowest-order ratio with maximum geometric coupling")
print("  - Higher ratios (4:3, 5:4, ...) have weaker coupling")
print("  - Lower ratios (2:1) exist but couple through a different mechanism")
print("    (period doubling, not node-antinode collision)")

In [ ]:
# ── Part 7: The experimental prediction ──────────────────────────────
#
# Setup: thin-walled tube, rosined tangential contact, variable
# rotation speed and contact pressure.
#
# The two-dimensional control surface (§1.2) maps directly:
#   - Rotation speed → drive (tangential velocity at contact)
#   - Contact pressure → coupling strength (normal force)
#
# Predictions:
#   1. Below a critical rotation speed (or above critical pressure),
#      the tube enters circumferential stick-slip
#   2. The dominant excited mode should be m=2 (lowest circumferential)
#   3. At deeper drive into the stick-slip regime, m=3 should activate
#   4. The m=3/m=2 frequency ratio should be 3:2
#   5. The mode pattern should be a TRAVELING wave (rotating around
#      the tube), not a standing wave anchored to the contact point
#   6. If both traveling, the frequency ratio 3:2 is a QPO analogue

print("=== Experimental Prediction: Tabletop QPO Generator ===")
print()
print("Setup:")
print("  - Thin-walled brass or aluminum tube (wall thickness << radius)")
print("  - Mounted on bearings or freely suspended at axial nodes")
print("  - Rosined felt pad pressed tangentially against outer surface")
print("  - Tube rotated at variable speed past the fixed pad")
print("  - OR: pad moved tangentially past a fixed tube")
print()
print("Measurement:")
print("  - Microphone → FFT of acoustic output")
print("  - Accelerometer on tube wall → mode shape identification")
print("  - Variable: rotation speed (drive) and pad pressure (coupling)")
print()
print("Predicted phase diagram:")
print()
print("  pressure")
print("    ↑")
print("    │  ┌─────────────────────┐")
print("    │  │  SUBHARMONIC REGIME  │")
print("    │  │  m=2 + m=3 excited   │")
print("    │  │  3:2 frequency ratio  │")
high = "    │  │  traveling wave (QPO) │"
print(high)
print("    │  └────────┐            │")
print("    │           │  NORMAL     │")
print("    │           │  m=2 only   │")
print("    │           │  (wine glass │")
print("    │           │   regime)   │")
print("    │           └────────────┘")
print("    │")
print("    └────────────────────────→ rotation speed")
print()
print("The critical test:")
print("  1. Start in the normal regime (fast rotation, moderate pressure)")
print("     → tube sings in m=2 mode (wine glass behavior)")
print("  2. Slow the rotation (or increase pressure)")
print("     → at threshold, m=3 mode activates")
print("     → frequency ratio between the two peaks should be 3:2")
print("  3. Check if the pattern rotates (traveling wave)")
print("     → if yes, this is a tabletop QPO")
print()
print("If this works, the string → cylinder → disk chain is validated:")
print("  same mechanism, same ratio, different geometry.")

In [ ]:
# ── Part 8: From cylinder to accretion disk ──────────────────────────
#
# An accretion disk is topologically an annulus — a cylinder unwrapped.
# But it has one crucial complication: differential rotation.
# The inner disk orbits faster than the outer disk.
#
# On a rigid cylinder, all points at a given angle have the same
# angular velocity. On a disk, the angular velocity is Keplerian:
#   Ω(r) ∝ r^{-3/2}
#
# This means the "drive" (relative velocity between adjacent annuli)
# is not uniform — it varies with radius. The stick-slip threshold
# is crossed at a specific radius where the shear rate matches
# the medium's critical value.

print("=== From Cylinder to Disk: What Differential Rotation Changes ===")
print()
print("Rigid cylinder:")
print("  - All circumferential points move together")
print("  - Drive is uniform: same tangential velocity everywhere")
print("  - Stick-slip activates everywhere at once")
print()
print("Accretion disk:")
print("  - Inner rings orbit faster than outer rings")
print("  - Drive = shear rate = r·dΩ/dr ∝ r^{-3/2}")
print("  - Stick-slip activates at a SPECIFIC RADIUS where")
print("    shear/threshold enters the critical band")
print()

# Show the shear profile
print("Keplerian shear rate vs radius (normalized):")
print()
print(f"  {'r/r_ISCO':>10s}  {'Ω(r)':>8s}  {'shear':>8s}  profile")
print("-" * 55)

for i in range(1, 21):
    r = 1.0 + i * 0.5  # r/r_ISCO
    omega = r ** (-1.5)  # Keplerian
    shear = 1.5 * r ** (-2.5)  # r·dΩ/dr
    
    bar_len = int(shear / 0.2)
    bar = "█" * min(bar_len, 40)
    
    if abs(r - 3.0) < 0.3:
        marker = " ◄ typical QPO radius"
    elif abs(r - 1.5) < 0.3:
        marker = " ◄ near ISCO"
    else:
        marker = ""
    
    print(f"  {r:10.1f}  {omega:8.3f}  {shear:8.3f}  {bar}{marker}")

print()
print("The shear rate (drive) decreases with radius.")
print("Near ISCO: shear is extreme → force-driven stick-slip (§2.3)")
print("Far out: shear is low → if low enough, slow-drive stick-slip (§2.1)")
print()
print("The disk crosses the stick-slip threshold at a specific radius.")
print("The azimuthal mode structure at that radius determines the QPO frequencies.")
print("The m=2 and m=3 modes at the threshold radius give the 3:2 ratio.")
print()
print("The mass-independence follows immediately:")
print("  The MODE NUMBERS (m=2, m=3) are integers.")
print("  The RATIO (3/2) is a ratio of integers.")
print("  Integers don't depend on the mass of the black hole.")
print("  The physical frequencies scale with mass; the ratio does not.")

## What this notebook shows

1. **The string-to-cylinder promotion preserves the 3:2 nodal geometry.**
   The node-antinode collision between modes 2 and 3 that produces the
   3:2 ratio on a string (§5.6) maps directly onto the circumferential
   modes of a cylinder. The geometry is scale-free AND topology-free.

2. **Cylinders support traveling waves; strings don't.** This is the
   key geometric promotion: on a cylinder, the stick-slip subharmonic
   can propagate as a traveling wave — a rotating pattern. On a string,
   it can only stand in place. A QPO IS a traveling subharmonic.

3. **Cylinder stick-slip already exists.** Wine glasses, singing bowls,
   and brake disks all exhibit circumferential stick-slip. The wine
   glass (rubbed rim) is a bowed cylinder that typically excites the
   m=2 mode. The question is whether it can be driven into the
   subharmonic regime to produce m=3/m=2 mode-locking at 3:2.

4. **The experimental prediction is sharp.** A thin-walled tube driven
   by tangential friction should produce circumferential stick-slip.
   At subharmonic onset (slow rotation or high pressure), the m=3 mode
   should activate alongside m=2, producing a 3:2 frequency ratio.
   If the pattern rotates (traveling wave), it is a tabletop QPO.

5. **The accretion disk is a differentially rotating cylinder.** The
   cylinder's uniform rotation becomes Keplerian shear, but the
   azimuthal mode structure is preserved. The 3:2 ratio is set by
   integer mode numbers, which are mass-independent.

### The chain, extended

String → cylinder → disk.
Standing subharmonic → traveling subharmonic → QPO.
Touch harmonic at 1/3, 2/3 → nodal meridians at 60°, 120° → azimuthal modes m=2, m=3.
Same geometry, same ratio, different topology.

---

*Uses only Python standard library. CC0.*